# 04 — Ajustar o comportamento sem substituir o RAG

O RAG da etapa 03 fornece conhecimento recuperado no momento da pergunta. QLoRA responde a outra necessidade: adaptar padrões de resposta usando exemplos supervisionados, treinando poucos parâmetros adicionais e mantendo congelada a base quantizada. O índice Chroma e o encoder permanecem os mesmos.

```mermaid
flowchart LR
 A[Qwen base em 4 bits] --> C[base congelada + adaptadores LoRA]
 B[exemplos revisados: pergunta → resposta] --> D[loss nos tokens da resposta]
 D --> C
 C --> E[adaptador salvo]
 E --> F[etapa 05: avaliar com/sem RAG]
```

O ajuste não deve ser tratado como memória confiável dos artigos. O experimento verificará se os exemplos melhoram explicação e comportamento sem degradar fundamentação.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: 04 — Fine-tuning QLoRA didático
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
from pathlib import Path
import json

RANDOM_STATE = 42
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_DOCUMENTOS = DIRETORIO_PROJETO / "documentos"
DIRETORIO_PROCESSADOS = DIRETORIO_PROJETO / "dados_processados"
DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_PROCESSADOS.mkdir(exist_ok=True)
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

## 1. Carregar uma base limpa

O treinamento deve começar com a mesma revisão do Qwen usada no baseline da etapa 00. Recarrega-se a base sem adaptador em um kernel limpo; isso evita continuar acidentalmente de pesos modificados em outra execução. Como o treino e a interface disputam a GPU, encerre a aplicação antes de carregar o modelo.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
import random
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

NOME_MODELO = "Qwen/Qwen2.5-3B-Instruct"
INSTRUCAO_SISTEMA = (
    "Responda em português, de forma didática e impessoal. "
    "Diferencie fatos, hipóteses e exemplos sintéticos. Não invente resultados."
)
set_seed(RANDOM_STATE)
tokenizador = AutoTokenizer.from_pretrained(NOME_MODELO)
modelo = AutoModelForCausalLM.from_pretrained(
    NOME_MODELO, device_map={"": 0}, dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
    ),
    attn_implementation="sdpa",
)
print("Base limpa carregada.")

## 2. Preparar exemplos e conferir a supervisão

O arquivo `dataset_sft.jsonl` contém perguntas e respostas didáticas separadas por famílias. O conjunto atual é um piloto para inspecionar o pipeline, pequeno demais para sustentar uma afirmação de especialização. PDFs não viram automaticamente pares de pergunta e resposta: cada resposta de treino precisa ser construída e revisada.

No treinamento causal, o modelo recebe tokens de prompt e resposta, mas a loss pode ser calculada somente sobre a resposta. O valor `-100` instrui a função de loss a ignorar posições do prompt e do padding. Isso ensina a continuação desejada sem penalizar o modelo por reproduzir a instrução. A célula mostra tokens e labels para verificar a máscara, em vez de presumir que o template fez isso corretamente.

---

### Loss causal e máscara de resposta

Uma LLM causal aprende a prever o próximo token. Para a sequência [pergunta, resposta A, resposta B, EOS], os logits em cada posição tentam prever o token seguinte. Com supervisão apenas da resposta, os labels do prompt recebem -100 (ignorados pela loss), enquanto A, B e EOS são supervisionados. O modelo continua vendo a pergunta como contexto, mas só é penalizado pela resposta produzida.

Exemplo esquemático:

| Posição | Pergunta | Resposta A | Resposta B | EOS |
|---|---:|---:|---:|---:|
| Label usado na loss | -100 | A | B | EOS |

A implementação causal desloca logits e labels internamente para alinhar previsão e próximo token. Deslocar os labels também manualmente causaria um deslocamento duplicado.

A forma resumida da entropia cruzada causal com máscara é:

$$
L=-\frac{1}{N}\sum_{t:y_t\ne -100}\log p_\theta(y_t\mid x_{<t}),
$$

em que N conta somente os tokens supervisionados. O prompt fornece contexto; a soma da loss inclui apenas os tokens cuja resposta correta foi mantida no label. O primeiro token da resposta é previsto usando a última posição do prompt.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Parte V — Fine-tuning QLoRA, passo a passo
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
exemplos = [json.loads(linha) for linha in (DIRETORIO_PROJETO / "dataset_sft.jsonl").read_text(encoding="utf-8").splitlines() if linha.strip()]
treino = [e for e in exemplos if e["split"] == "treino"]
validacao = [e for e in exemplos if e["split"] == "validacao"]
teste = [e for e in exemplos if e["split"] == "teste"]
print("Treino/validação/teste:", len(treino), len(validacao), len(teste))
assert set(e["familia"] for e in treino).isdisjoint(e["familia"] for e in validacao + teste)
print(treino[0])

lotes = {"treino": [], "validacao": []}
for nome_split, registros in [("treino", treino), ("validacao", validacao)]:
    for exemplo in registros:
        mensagens = [{"role": "system", "content": INSTRUCAO_SISTEMA}, {"role": "user", "content": exemplo["pergunta"]}]
        prefixo = tokenizador.apply_chat_template(mensagens, tokenize=True, add_generation_prompt=True)
        completo = tokenizador.apply_chat_template(mensagens + [{"role": "assistant", "content": exemplo["resposta"]}], tokenize=True)
        assert completo[:len(prefixo)] == prefixo
        if len(completo) > 512:
            raise ValueError(f"Exemplo longo: {exemplo['familia']}; revisar antes do treino.")
        labels = [-100] * len(prefixo) + completo[len(prefixo):]
        assert any(x != -100 for x in labels)
        lotes[nome_split].append({"input_ids": torch.tensor([completo]), "attention_mask": torch.ones(1, len(completo), dtype=torch.long), "labels": torch.tensor([labels])})
print("Shape:", lotes["treino"][0]["input_ids"].shape)
print("Labels:", lotes["treino"][0]["labels"])
print("Texto supervisionado:", tokenizador.decode([x for x in lotes["treino"][0]["labels"][0].tolist() if x != -100]))

## 3. Congelar a base e inserir LoRA

Uma camada linear usa a matriz de pesos W. LoRA mantém W congelada e aprende uma atualização de baixo posto; a equação e um exemplo de contagem aparecem abaixo.

QLoRA combina a base quantizada com esses adaptadores treináveis. Menos parâmetros treináveis reduz parte do custo, mas ativações e estados do otimizador continuam ocupando memória. A célula de treino fica protegida por uma flag: primeiro revise exemplos e shapes, depois habilite o piloto conscientemente.

---

### LoRA: atualização de baixo posto

Para uma matriz de pesos W com dimensão d_out × d_in, LoRA aprende duas matrizes menores A e B e soma a atualização escalada:

$$
W'=W+\frac{\alpha}{r}BA,
\qquad
A\in\mathbb{R}^{r\times d_{in}},
\quad B\in\mathbb{R}^{d_{out}\times r}.
$$

Exemplo: uma matriz 1024 × 1024 tem 1.048.576 pesos. Com rank r = 8, A e B juntas têm 8×1024 + 1024×8 = 16.384 parâmetros treináveis, cerca de 1,56% dos pesos dessa matriz. O cálculo não inclui todas as camadas nem significa que o consumo total de memória caia nessa mesma proporção.

### O que QLoRA acrescenta

Na QLoRA, os pesos da base ficam quantizados e congelados; os adaptadores LoRA são treinados. Quantização reduz memória dos pesos, e LoRA reduz parâmetros atualizados. Ativações e estados do otimizador ainda consomem memória, por isso batch, sequência e acumulação precisam ser medidos.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
EXECUTAR_AJUSTE = False  # Alterar para True após revisar os exemplos e encerrar o chat.
if EXECUTAR_AJUSTE:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if hasattr(modelo, "peft_config"):
        raise RuntimeError("Recarregue a base limpa antes de repetir o ajuste.")
    modelo.config.use_cache = False
    modelo = prepare_model_for_kbit_training(modelo, use_gradient_checkpointing=True)
    config_lora = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM",
        base_model_name_or_path=NOME_MODELO,
        revision=getattr(modelo.config, "_commit_hash", None),
    )
    modelo = get_peft_model(modelo, config_lora)
    modelo.print_trainable_parameters()
else:
    print("Ajuste desativado. Nenhum peso foi treinado.")

## 4. Treinar, validar e salvar o melhor adaptador

O loop torna visível o que um trainer costuma encapsular: forward, cálculo da loss, backward, acumulação de gradientes, atualização do otimizador e avaliação. Com batch pequeno, acumular gradientes aproxima um batch efetivo maior sem guardar todos os exemplos de uma vez.

A loss de validação acompanha exemplos que não atualizam os pesos. O adaptador salvo é escolhido por essa validação, nunca pelo conjunto de teste. Com apenas poucas famílias e poucos exemplos, diferenças de loss são sinais para depuração, não evidência de ganho confiável. Após o salvamento, reinicie e recarregue base + adaptador para verificar que os artefatos podem ser usados fora do processo de treino.

---

### Acumulação e escolha do checkpoint

Com batch por dispositivo 1 e acumulação por 4 passos, o otimizador atualiza após quatro microbatches; o batch efetivo é aproximadamente 4. Isso reduz memória por passo, embora cada época tenha mais passos. Se a última janela tiver apenas dois microbatches, a loss deve ser normalizada por 2, não por 4.

A validação mede exemplos que não atualizam os pesos. Exemplo ilustrativo: losses de validação por época 0,90, 0,72 e 0,78 indicariam salvar a época 2. O conjunto de teste fica fora dessa escolha. Loss menor é útil para acompanhar o objetivo de treino, mas não prova que citações ou explicações ficaram melhores.

In [ ]:
# -----------------------------------------------------------------------------
# Objetivo desta célula: Pergunta de aprendizagem: qual transformação ou decisão este bloco permite compreender?
# O código abaixo mantém as etapas explícitas para permitir inspeção de entradas,
# transformações e saídas. Conferir os valores impressos antes de avançar.
# -----------------------------------------------------------------------------
if EXECUTAR_AJUSTE:
    import random
    set_seed(RANDOM_STATE)
    otimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=1e-4)
    EPOCAS = 3
    ACUMULACAO = 4
    melhor_loss = float("inf")
    historico_treino = []
    for epoca in range(EPOCAS):
        modelo.train()
        ordem = list(range(len(lotes["treino"])))
        random.Random(RANDOM_STATE + epoca).shuffle(ordem)
        perdas = []
        for inicio in range(0, len(ordem), ACUMULACAO):
            grupo = ordem[inicio:inicio + ACUMULACAO]
            otimizador.zero_grad(set_to_none=True)
            for indice in grupo:
                batch = {k:v.to("cuda:0") for k,v in lotes["treino"][indice].items()}
                saida = modelo(**batch)
                if not torch.isfinite(saida.loss):
                    raise RuntimeError("Loss não finita. Interromper e investigar precisão/dados.")
                perdas.append(saida.loss.item())
                (saida.loss / len(grupo)).backward()
            torch.nn.utils.clip_grad_norm_([p for p in modelo.parameters() if p.requires_grad], 1.0)
            otimizador.step()
        modelo.eval()
        soma_loss, total_tokens = 0.0, 0
        with torch.no_grad():
            for lote in lotes["validacao"]:
                batch = {k:v.to("cuda:0") for k,v in lote.items()}
                quantidade = int((batch["labels"][:, 1:] != -100).sum())
                soma_loss += modelo(**batch).loss.item() * quantidade
                total_tokens += quantidade
        loss_val = soma_loss / total_tokens
        registro_epoca = {"epoca": epoca+1, "loss_treino": sum(perdas)/len(perdas), "loss_validacao": loss_val}
        historico_treino.append(registro_epoca)
        print(registro_epoca)
        if loss_val < melhor_loss:
            melhor_loss = loss_val
            modelo.save_pretrained(DIRETORIO_PROJETO / "adaptador_pf")
            tokenizador.save_pretrained(DIRETORIO_PROJETO / "adaptador_pf")
    (DIRETORIO_RESULTADOS / "historico_qlora.json").write_text(json.dumps(historico_treino, indent=2), encoding="utf-8")
    print("Melhor adaptador salvo. Reiniciar o kernel/servidor para carregá-lo na inferência.")